# Commander Curriculum Training
Upload the locally prepared `cedhai-colab.zip`. No GitHub token is needed. This runs scripted behavior cloning and outcome prediction, not self-play RL. One seed per split is sufficient only for a wiring smoke test.

Run cells in order. CPU is sufficient for this tiny model; optionally select a GPU runtime. Checkpoints are saved to your Google Drive after each epoch. Drive mounting requires your authorization. Do not upload an archive from an untrusted source: it contains Python code that will execute.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one cedhai-colab.zip archive')
archive_path = next(iter(uploaded))


In [ ]:
import hashlib, json, zipfile
from pathlib import Path, PurePosixPath
workspace = Path('/content/cedhai-training')
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('Duplicate archive entries')
    metadata = json.loads(archive.read('bundle.json'))
    if metadata['schema_version'] != 1 or set(names) != set(metadata['sha256']) | {'bundle.json'}:
        raise ValueError('Invalid bundle manifest')
    for name, expected in metadata['sha256'].items():
        parts = PurePosixPath(name).parts
        if not parts or parts[0] not in ('project', 'dataset') or '..' in parts or '\\' in name:
            raise ValueError('Unsafe archive path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError('Bundle checksum mismatch')
        target = workspace.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print('Source revision:', metadata['git_revision'])


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(workspace / 'project') + '[training]'], check=True)
subprocess.run([sys.executable, '-m', 'cedhai.dataset_report', '--dataset', str(workspace / 'dataset')], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
output_root = Path('/content/drive/MyDrive/cedhai/checkpoints')
output_root.mkdir(parents=True, exist_ok=True)
run_dir = output_root / datetime.now(timezone.utc).strftime('smoke-%Y%m%d-%H%M%S-%f')
EPOCHS = 5
SEED = 0
# Set this to a prior latest.pt to continue the SAME bundle/configuration.
# EPOCHS is the total desired epoch count, not additional epochs.
RESUME = None


In [ ]:
command = [sys.executable, '-m', 'cedhai.train', '--dataset', str(workspace / 'dataset'),
           '--out', str(run_dir), '--epochs', str(EPOCHS), '--seed', str(SEED), '--device', 'auto']
if RESUME:
    command += ['--resume', str(RESUME)]
subprocess.run(command, check=True)
print('Persistent checkpoint:', run_dir / 'latest.pt')


In [ ]:
report = json.loads((run_dir / 'metrics.json').read_text())
print(json.dumps({'device': report['device'], 'torch_version': report['torch_version'],
                  'baseline': report['baseline'], 'last_epoch': report['history'][-1],
                  'warnings': report['warnings']}, indent=2))
files.download(str(run_dir / 'latest.pt'))


## Evaluate Locally
The downloaded `latest.pt` is compatible with `cedhai.model.ModelAgent`. Play a curriculum game using `cedhai.play_model --checkpoint <path> --out <new run folder>`. Imitation accuracy is agreement with scripts, not win rate. GPU and CPU results may differ; the saved configuration, dataset hash, and PyTorch version record the experiment.
